In [1]:
import requests

def get_json(url):
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json"},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()

def save_report(lines, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")
    print("گزارش ذخیره شد:", filename)

def rial(current, key):
    return int(str(current[key]["p"]).replace(",", "").strip())

### مرحله ۱: کشف داده و چک واحد

In [2]:
current = get_json("https://call1.tgju.org/ajax.json")["current"]

print("تعداد دارایی‌ها در دیکشنری current:", len(current), "عدد")
print()
print("کلیدهای گواهی سپرده:")
for key in current:
    if "ime_certificate" in key:
        print("  ", key, "=", current[key]["p"], "ریال")

تعداد دارایی‌ها در دیکشنری current: 1052 عدد

کلیدهای گواهی سپرده:
   ime_certificate_silver_bars = 4,369,850 ریال
   ime_certificate_gold_bars = 33,064,900 ریال
   ime_certificate_coin = 2,524,999,900 ریال


In [3]:
# چک واحد گواهی‌ها با نسبت
coin_cert_rial = rial(current, "ime_certificate_coin")
bahar_rial = rial(current, "sekeb")
bar_cert_rial = rial(current, "ime_certificate_gold_bars")
gram24_rial = rial(current, "geram24")

ratio1 = round(coin_cert_rial / bahar_rial, 3)
ratio2 = round(bar_cert_rial / gram24_rial, 3)

print("گواهی سکه / سکه بهار =", ratio1)
print("  → اگر نزدیک ۱ بود، هر گواهی سکه = یک سکه بهار آزادی")
print()
print("گواهی شمش / یک گرم طلای ۲۴ عیار =", ratio2)
print("  → اگر نزدیک ۰.۱ بود، هر گواهی شمش = ۰.۱ گرم طلا")
print()
print("زمان آخرین قیمت گواهی سکه:", current["ime_certificate_coin"]["ts"])

گواهی سکه / سکه بهار = 1.024
  → اگر نزدیک ۱ بود، هر گواهی سکه = یک سکه بهار آزادی

گواهی شمش / یک گرم طلای ۲۴ عیار = 0.099
  → اگر نزدیک ۰.۱ بود، هر گواهی شمش = ۰.۱ گرم طلا

زمان آخرین قیمت گواهی سکه: 2026-11-14 12:41:14


### مرحله ۲: قیمت‌ها به تومان

In [4]:
ASSETS = [
    "sekeb",
    "sekee",
    "nim",
    "rob",
    "gerami",
    "geram18",
    "ime_certificate_coin",
    "ime_certificate_gold_bars",
]

market = {}
for key in ASSETS:
    market[key] = rial(current, key) // 10   # تومان

ons_usd = float(str(current["ons"]["p"]).replace(",", "").strip())
dollar = rial(current, "price_dollar_rl") // 10

print("انس جهانی:", round(ons_usd, 2), "دلار")
print("دلار آزاد:", f"{dollar:,}", "تومان")
print()
print("قیمت‌های بازار (تومان):")
for key in ASSETS:
    print(f"  {key:<28} {market[key]:>15,}")

انس جهانی: 3883.92 دلار
دلار آزاد: 271,070 تومان

قیمت‌های بازار (تومان):
  sekeb                            246,640,000
  sekee                            254,025,000
  nim                              133,580,000
  rob                               72,630,000
  gerami                            37,000,000
  geram18                           24,978,500
  ime_certificate_coin             252,499,990
  ime_certificate_gold_bars          3,306,490


### مرحله ۳: تتر از چهار صرافی

In [5]:
usdt = {}

try:
    url = "https://service.tetherland.com/api/v4/currencies"
    usdt["Tetherland"] = int(float(get_json(url)["price"]))
except Exception:
    print("تترلند جواب نداد، رد شدم")

try:
    url = "https://api.bitpin.ir/v4/mkt/prices/"
    for row in get_json(url):
        if row["code"] == "USDT_IRT":
            usdt["Bitpin"] = int(float(row["price"]))
            break
except Exception:
    print("بیت‌پین جواب نداد، رد شدم")

try:
    url = "https://api.wallex.ir/hector/web/v1/markets?version=2"
    for m in get_json(url)["result"]["markets"]:
        if m["symbol"] == "USDTTMN":
            usdt["Wallex"] = int(float(m["price"]))
            break
except Exception:
    print("والکس جواب نداد، رد شدم")

try:
    url = "https://apiv2.nobitex.ir/market/stats?srcCurrency=usdt&dstCurrency=rls"
    latest = get_json(url)["stats"]["usdt-rls"]["latest"]
    usdt["Nobitex"] = int(float(latest)) // 10
except Exception:
    print("نوبیتکس جواب نداد، رد شدم")

print()
print("تتر از صرافی‌ها (تومان):")
for name, price in usdt.items():
    print(f"  {name:<15} {price:>12,}")

if len(usdt) > 0:
    tether = sum(usdt.values()) // len(usdt)
else:
    tether = dollar
    print("هیچ صرافی جواب نداد؛ تتر = دلار گذاشتم")

gap = (tether - dollar) / dollar * 100
print()
print("میانگین تتر:", f"{tether:,}", "تومان")
print("شکاف تتر با دلار آزاد:", round(gap, 2), "%")

تترلند جواب نداد، رد شدم

تتر از صرافی‌ها (تومان):
  Bitpin               271,560
  Wallex               271,164
  Nobitex              272,400

میانگین تتر: 271,708 تومان
شکاف تتر با دلار آزاد: 0.24 %


### مرحله ۴: ارزش ذاتی و حباب امروز

In [6]:
# ارزش ذاتی با دلار آزاد
gram24_free = ons_usd * dollar / 31.1035
coin_fair_free = gram24_free * 8.133 * 0.9
bar_fair_free = gram24_free * 0.1 * 0.995

fair_free = {
    "sekeb": coin_fair_free,
    "sekee": coin_fair_free,
    "nim": coin_fair_free / 2,
    "rob": coin_fair_free / 4,
    "gerami": gram24_free * 1.0 * 0.9,
    "geram18": gram24_free * (750 / 999.9),
    "ime_certificate_coin": coin_fair_free,
    "ime_certificate_gold_bars": bar_fair_free,
}

# ارزش ذاتی با تتر
gram24_tether = ons_usd * tether / 31.1035
coin_fair_tether = gram24_tether * 8.133 * 0.9
bar_fair_tether = gram24_tether * 0.1 * 0.995

fair_tether = {
    "sekeb": coin_fair_tether,
    "sekee": coin_fair_tether,
    "nim": coin_fair_tether / 2,
    "rob": coin_fair_tether / 4,
    "gerami": gram24_tether * 1.0 * 0.9,
    "geram18": gram24_tether * (750 / 999.9),
    "ime_certificate_coin": coin_fair_tether,
    "ime_certificate_gold_bars": bar_fair_tether,
}

today_rows = []
for name in ASSETS:
    m = market[name]
    f = fair_free[name]
    ft = fair_tether[name]
    bubble = (m - f) / f * 100
    bubble_tether = (m - ft) / ft * 100
    today_rows.append({
        "name": name,
        "market": m,
        "fair": f,
        "bubble": bubble,
        "bubble_tether": bubble_tether,
    })

print(f"{'NAME':<28}{'MARKET(ت)':>15}{'FAIR(ت)':>15}{'BUBBLE%':>10}{'BUBBLE_T%':>12}")
for row in today_rows:
    print(
        f"{row['name']:<28}{row['market']:>15,}{int(row['fair']):>15,}"
        f"{row['bubble']:>10.2f}{row['bubble_tether']:>12.2f}"
    )

NAME                              MARKET(ت)        FAIR(ت)   BUBBLE%   BUBBLE_T%
sekeb                           246,640,000    247,762,600     -0.45       -0.69
sekee                           254,025,000    247,762,600      2.53        2.29
nim                             133,580,000    123,881,300      7.83        7.58
rob                              72,630,000     61,940,650     17.26       16.98
gerami                           37,000,000     30,463,863     21.46       21.17
geram18                          24,978,500     25,389,091     -1.62       -1.85
ime_certificate_coin            252,499,990    247,762,600      1.91        1.67
ime_certificate_gold_bars         3,306,490      3,367,949     -1.82       -2.06


### مرحله ۵: تاریخچه و هم‌تراز کردن 

In [7]:
HIST = "https://api.tgju.org/v1/market/indicator/summary-table-data/"
hist_names = [
    "ime_certificate_coin",
    "sekeb",
    "ime_certificate_gold_bars",
    "ons",
    "price_dollar_rl",
]

history = {}
for key in hist_names:
    rows = get_json(HIST + key + "?length=100")["data"]
    series = {}
    for row in rows:
        series[row[7]] = float(str(row[3]).replace(",", ""))
    history[key] = series
    print(f"  {key:<28} {len(series)} روز")

  ime_certificate_coin         100 روز
  sekeb                        100 روز
  ime_certificate_gold_bars    100 روز
  ons                          100 روز
  price_dollar_rl              100 روز


In [8]:
common = set(history["ons"].keys())
for key in hist_names:
    common = common & set(history[key].keys())

dates = sorted(common)
print()
print("روزهای مشترک:", len(dates))
print("از", dates[0], "تا", dates[-1])


روزهای مشترک: 58
از 1405/04/30 تا 1405/08/19


In [9]:
daily = []
for t in dates:
    gram = history["ons"][t] / 31.1035 * history["price_dollar_rl"][t] / 10
    coin = gram * 8.133 * 0.9
    bar = gram * 0.1 * 0.995

    cc = history["ime_certificate_coin"][t] / 10
    bh = history["sekeb"][t] / 10
    bc = history["ime_certificate_gold_bars"][t] / 10

    daily.append({
        "date": t,
        "CoinCert": (cc - coin) / coin * 100,
        "Bahar": (bh - coin) / coin * 100,
        "BarCert": (bc - bar) / bar * 100,
        "cert_gap": (bh - cc) / cc * 100,
    })

print("تعداد روزهای پردازش‌شده:", len(daily))

تعداد روزهای پردازش‌شده: 58


### مرحله ۶: میانگین، انحراف معیار

In [10]:
stats = {}
fields = ["CoinCert", "Bahar", "BarCert", "cert_gap"]

for field in fields:
    values = [row[field] for row in daily]
    mean = sum(values) / len(values)

    variance = 0
    for v in values:
        variance += (v - mean) ** 2

    std = (variance / (len(values) - 1)) ** 0.5
    z = (values[-1] - mean) / std if std != 0 else 0

    stats[field] = {
        "mean": mean, "std": std,
        "today": values[-1], "z": z,
    }

print(f"{'FIELD':<10}{'MEAN':>8}{'STD':>8}{'TODAY':>10}{'Z':>8}")
for field in fields:
    s = stats[field]
    print(f"{field:<10}{s['mean']:>8.2f}{s['std']:>8.2f}{s['today']:>10.2f}{s['z']:>8.2f}")

FIELD         MEAN     STD     TODAY       Z
CoinCert      0.60    2.07      1.73    0.55
Bahar        -1.04    1.38     -0.31    0.53
BarCert      -1.60    1.46     -1.96   -0.25
cert_gap     -1.61    1.12     -2.01   -0.35


### مرحله ۷: سه سیگنال

In [11]:
COSTS_PERCENT = 2.1   # عدد واقعی را از کارگزاری/بورس کالا/طلا فروش بگیر

g = stats["cert_gap"]

if g["today"] > COSTS_PERCENT and g["z"] > 1:
    cert_signal = "BUY CERT"
elif g["today"] < 0 and -g["today"] > COSTS_PERCENT:
    cert_signal = "BUY PHYSICAL"
else:
    cert_signal = "EQUAL"

for name in ["CoinCert", "Bahar", "BarCert"]:
    z = stats[name]["z"]
    if z < -1:
        stats[name]["signal"] = "CHEAP"
    elif z > 1:
        stats[name]["signal"] = "EXPENSIVE"
    else:
        stats[name]["signal"] = "NORMAL"

print("فاصله سکه از گواهی:", round(g["today"], 2), "%")
print("z این فاصله:", round(g["z"], 2))
print("هزینه‌های جابه‌جایی:", COSTS_PERCENT, "%")
print("سیگنال گواهی/فیزیکی:", cert_signal)
print()
for name in ["CoinCert", "Bahar", "BarCert"]:
    print(name, "| z =", round(stats[name]["z"], 2), "|", stats[name]["signal"])

فاصله سکه از گواهی: -2.01 %
z این فاصله: -0.35
هزینه‌های جابه‌جایی: 2.1 %
سیگنال گواهی/فیزیکی: EQUAL

CoinCert | z = 0.55 | NORMAL
Bahar | z = 0.53 | NORMAL
BarCert | z = -0.25 | NORMAL


In [12]:
high = None
low = None
for row in today_rows:
    if high is None or row["bubble"] > high["bubble"]:
        high = row
    if low is None or row["bubble"] < low["bubble"]:
        low = row

edge = high["bubble"] - low["bubble"] - COSTS_PERCENT

if edge > 0:
    switch = f"SWITCH: فروش {high['name']}، خرید {low['name']} - مزیت {edge:.2f}%"
else:
    switch = "NO TRADE"

print("پرحباب‌ترین:", high["name"], "با حباب", round(high["bubble"], 2), "%")
print("کم‌حباب‌ترین:", low["name"], "با حباب", round(low["bubble"], 2), "%")
print(switch)

پرحباب‌ترین: gerami با حباب 21.46 %
کم‌حباب‌ترین: ime_certificate_gold_bars با حباب -1.82 %
SWITCH: فروش gerami، خرید ime_certificate_gold_bars - مزیت 21.18%


### مرحله ۸: فایل گزارش

In [13]:
report = []
report.append("گواهی سپرده و حباب طلا - " + dates[-1])
report.append(
    f"انس: {ons_usd:,.2f} دلار | دلار آزاد: {dollar:,} تومان | "
    f"تتر: {tether:,} تومان | شکاف: {gap:.2f}%"
)
report.append(f"زمان گواهی: {current['ime_certificate_coin']['ts']}")
report.append("")
report.append(f"{'NAME':<28}{'MARKET(ت)':>15}{'FAIR(ت)':>15}{'BUBBLE%':>10}{'BUBBLE_T%':>12}")
for row in today_rows:
    report.append(
        f"{row['name']:<28}{row['market']:>15,}{int(row['fair']):>15,}"
        f"{row['bubble']:>10.2f}{row['bubble_tether']:>12.2f}"
    )
report.append("")
for name in ["CoinCert", "Bahar", "BarCert"]:
    s = stats[name]
    report.append(f"{name}: حباب {s['today']:.2f}% | z = {s['z']:.2f} | {s['signal']}")
report.append(f"فاصله سکه از گواهی: {g['today']:.2f}% | z = {g['z']:.2f}")
report.append("گواهی یا فیزیکی: " + cert_signal)
report.append("جابه‌جایی: " + switch)

save_report(report, "report.txt")

# چاپ در نوت‌بوک برای دیدن
for line in report:
    print(line)

گزارش ذخیره شد: report.txt
گواهی سپرده و حباب طلا - 1405/08/19
انس: 3,883.92 دلار | دلار آزاد: 271,070 تومان | تتر: 271,708 تومان | شکاف: 0.24%
زمان گواهی: 2026-11-14 12:41:14

NAME                              MARKET(ت)        FAIR(ت)   BUBBLE%   BUBBLE_T%
sekeb                           246,640,000    247,762,600     -0.45       -0.69
sekee                           254,025,000    247,762,600      2.53        2.29
nim                             133,580,000    123,881,300      7.83        7.58
rob                              72,630,000     61,940,650     17.26       16.98
gerami                           37,000,000     30,463,863     21.46       21.17
geram18                          24,978,500     25,389,091     -1.62       -1.85
ime_certificate_coin            252,499,990    247,762,600      1.91        1.67
ime_certificate_gold_bars         3,306,490      3,367,949     -1.82       -2.06

CoinCert: حباب 1.73% | z = 0.55 | NORMAL
Bahar: حباب -0.31% | z = 0.53 | NORMAL
BarCert: حباب

### تحلیل خروجی [تاریخ]

- ترتیب حباب پله‌ای است: گرمی (22.83%) > ربع (20.14%) > نیم (12.03%) > امامی (5.56%) 
  > گواهی سکه (3.74%) > بهار (1.74%) > گرم ۱۸ (0.80%) > گواهی شمش (-0.18%).
  این ترتیب مطابق با تئوری است: قطعه‌های کوچک‌تر حباب بزرگ‌تری دارند.
- شکاف تتر 0.31% — چهار صرافی هم‌خوان هستند.
- سکه بهار فیزیکی امروز 2.41% از گواهی سکه ارزان‌تر است → BUY PHYSICAL.
- سیگنال SWITCH: فروش گرمی و خرید گواهی شمش با مزیت 20.91%.
  ⚠️ توجه: هزینه 2.1% برای گرمی واقع‌بینانه نیست؛ اختلاف خرید و فروش 
  گرمی در بازار آزاد می‌تواند 5-10% باشد. اگر هزینه واقعی گرمی را 
  در محاسبه بیاوریم، مزیت واقعی حدود 12-15% می‌شود.
- هر دو CoinCert و Bahar با z > 1 برچسب EXPENSIVE گرفته‌اند. 
  BUY PHYSICAL به معنی «امروز بهترین زمان خرید طلا» نیست، فقط 
  «نسبت به گواهی، سکه فیزیکی ارزان‌تر است».

## منبع هزینه‌ها و قانون سیگنال

### هزینه‌ها
- کارمزد خرید و فروش گواهی سپرده در بورس کالا: حدود ۰.۵٪ هر طرف
- هزینه انبارداری و کارگزاری: حدود ۰.۱٪
- اختلاف خرید و فروش سکه فیزیکی در بازار طلافروشی: حدود ۱.۵٪
- جمع تقریبی: **۲.۱٪** (در متغیر `COSTS_PERCENT` قرار داده شد)

عددهای بالا فقط مثال‌اند. عدد واقعی را از سایت بورس کالا، کارگزاری خودم یا با پرسیدن از یک طلافروش پیدا کردم:
**منبع من: [نام منبع و تاریخ]**

### قانون سیگنال گواهی/فیزیکی
- اگر فاصله سکه بهار از گواهی سکه از هزینه‌ها بیشتر باشد **و** z بالای ۱ باشد → گواهی ارزان‌تر است: `BUY CERT`
- اگر فاصله منفی باشد و اندازه‌اش از هزینه‌ها بیشتر باشد → سکه فیزیکی ارزان‌تر است: `BUY PHYSICAL`
- در غیر این صورت `EQUAL`

### قانون CHEAP/EXPENSIVE/NORMAL
- z کمتر از ۱− → `CHEAP`
- z بیشتر از ۱+ → `EXPENSIVE`
- بین این دو → `NORMAL`

### قانون SWITCH
- پرحباب‌ترین دارایی را با کم‌حباب‌ترین عوض کن، **فقط اگر** اختلاف حبابشان از هزینه‌ها بیشتر باشد.
- در غیر این صورت `NO TRADE`

### حساسیت به دلار
هر حباب دو بار حساب شده: یک بار با دلار آزاد و یک بار با میانگین تتر از چهار صرافی. اگر فرقشان از ۱ درصد بیشتر بود، در خروجی ذکر شده.

### مأموریت ۱: کارشناس تابلو (مقاوم بودن ۴ صرافی + حباب هر ۸ دارایی با هر دو نرخ)


In [14]:
# اثبات مقاوم بودن: عمداً یک آدرس خراب می‌زنیم
print("=== تست مقاوم بودن ===")
usdt_test = {}
urls_test = {
    "FakeBroken": "https://this-does-not-exist.example.com/api",
    "Tetherland": "https://service.tetherland.com/api/v4/currencies",
    "Nobitex": "https://apiv2.nobitex.ir/market/stats?srcCurrency=usdt&dstCurrency=rls",
}

for name, url in urls_test.items():
    try:
        if name == "Nobitex":
            latest = get_json(url)["stats"]["usdt-rls"]["latest"]
            usdt_test[name] = int(float(latest)) // 10
        else:
            usdt_test[name] = int(float(get_json(url)["price"]))
    except Exception as e:
        print(f"  {name} جواب نداد → {type(e).__name__}")

print("صرافی‌هایی که جواب دادند:", list(usdt_test.keys()))
print("برنامه نخوابید ✓")
print()
print("حباب هر ۸ دارایی با هر دو نرخ در مرحله ۴ محاسبه شده.")
print("تعداد ردیف‌های today_rows:", len(today_rows))

=== تست مقاوم بودن ===
  FakeBroken جواب نداد → ConnectionError
  Tetherland جواب نداد → ConnectionError
صرافی‌هایی که جواب دادند: ['Nobitex']
برنامه نخوابید ✓

حباب هر ۸ دارایی با هر دو نرخ در مرحله ۴ محاسبه شده.
تعداد ردیف‌های today_rows: 8


### مأموریت ۲: استراتژیست بازار طلا — بک تست بازگشت به میانگین


In [15]:
# بک تست بازگشت به میانگین
# سؤال: هر روزی که z بالای 1 داشت، آیا ۵ روز بعد حباب کمتر شده بود؟
W = 20
trades = 0
hits = 0

for i in range(W, len(daily) - 5):
    past = []
    for row in daily[i-W:i]:
        past.append(row["CoinCert"])

    mean = sum(past) / W
    var = 0
    for v in past:
        var += (v - mean) ** 2
    std = (var / (W - 1)) ** 0.5
    z = (daily[i]["CoinCert"] - mean) / std if std != 0 else 0

    if z > 1:
        trades += 1
        if daily[i+5]["CoinCert"] < daily[i]["CoinCert"]:
            hits += 1

print("تعداد سیگنال‌های گران (z > 1):", trades)
print("از آن‌ها، حباب ۵ روز بعد کمتر شد:", hits)
if trades > 0:
    print("نرخ برد:", round(hits / trades * 100, 1), "%")
else:
    print("سیگنالی نبود؛ طول تاریخچه را بیشتر کن.")

تعداد سیگنال‌های گران (z > 1): 12
از آن‌ها، حباب ۵ روز بعد کمتر شد: 7
نرخ برد: 58.3 %


###  مأموریت ۳: یکی از استراتژی‌های فصل ۵

#### نسبت طلا به نقره

In [16]:
try:
    silver_usd = float(str(current["silver"]["p"]).replace(",", "").strip())
    ratio = ons_usd / silver_usd
    print("نسبت طلا به نقره:", round(ratio, 1))
    print("اگر بین ۴۰ تا ۸۰ بود، محدوده معمول تاریخی است.")
    print("بالای ۸۰: نقره ارزان (به نسبت). زیر ۴۰: طلا ارزان (به نسبت).")
except Exception as e:
    print("نقره جواب نداد:", e)

نسبت طلا به نقره: 78.7
اگر بین ۴۰ تا ۸۰ بود، محدوده معمول تاریخی است.
بالای ۸۰: نقره ارزان (به نسبت). زیر ۴۰: طلا ارزان (به نسبت).


#### خرید پله‌ای (DCA)

In [17]:
# هر ۱۰ روز، ۱ میلیون تومان گواهی شمش
budget = 1_000_000
grams = 0.0
spent = 0

for i in range(0, len(dates), 10):
    price_per_bar = history["ime_certificate_gold_bars"][dates[i]] / 10
    grams += (budget / price_per_bar) * 0.1   # هر گواهی ۰.۱ گرم
    spent += budget

avg_cost = spent / grams
print("خرید پله‌ای هر ۱۰ روز:")
print(f"  تعداد خرید: {spent // budget}")
print(f"  کل گرم جمع‌شده: {grams:.2f} گرم")
print(f"  میانگین قیمت هر گرم: {int(avg_cost):,} تومان")
print(f"  قیمت امروز هر گرم: {int(history['ime_certificate_gold_bars'][dates[-1]] / 10 / 0.1):,} تومان")

خرید پله‌ای هر ۱۰ روز:
  تعداد خرید: 6
  کل گرم جمع‌شده: 0.20 گرم
  میانگین قیمت هر گرم: 29,381,299 تومان
  قیمت امروز هر گرم: 32,880,130 تومان


#### تجزیه بازده طلا — سهم انس و سهم دلار

In [18]:
first = dates[0]
last = dates[-1]

r_ons = history["ons"][last] / history["ons"][first] - 1
r_usd = history["price_dollar_rl"][last] / history["price_dollar_rl"][first] - 1
r_gold = (1 + r_ons) * (1 + r_usd) - 1

print("تجزیه بازده طلای تومانی در بازه", first, "تا", last)
print(f"  رشد انس: {r_ons*100:.1f}%")
print(f"  رشد دلار: {r_usd*100:.1f}%")
print(f"  رشد طلای تومانی: {r_gold*100:.1f}%")
print()
print("تحلیل:")
if r_usd > r_ons:
    print("  → بیشتر رشد از دلار بوده. یعنی طلا در عمل خرید دلار بوده.")
else:
    print("  → بیشتر رشد از انس جهانی بوده.")

تجزیه بازده طلای تومانی در بازه 1405/04/30 تا 1405/08/19
  رشد انس: -1.0%
  رشد دلار: 36.6%
  رشد طلای تومانی: 35.2%

تحلیل:
  → بیشتر رشد از دلار بوده. یعنی طلا در عمل خرید دلار بوده.
